<p>
<font size='5' face='Georgia, Arial'>IIC2233 Apunte Programación Avanzada</font><br>
<font size='1'> Modificado en 2026-1 al 2026-2 por Equipo Docente IIC2233. </font>
</p>

# Tabla de contenidos

1. [Consultas concurrentes](#consultas-concurrentes)
    1. [ThreadPoolExecutor](#threadpool)
    2. [Comparación de tiempos](#tiempos)
2. [Condiciones de carrera](#race-condition)
3. [Riesgos de concurrencia en el cliente](#riesgos-cliente)
    1. [Solicitudes duplicadas](#duplicadas)
    2. [Rate limiting y reintentos](#rate-limit)
4. [Condiciones de carrera en servidores](#race-servidor)
    1. [Estado compartido](#estado-compartido)
    2. [Lock y sus límites](#lock)

<a id="consultas-concurrentes"></a>
## Consultas concurrentes

Hasta ahora enviamos una solicitud y esperamos su respuesta antes de continuar. Para trabajo limitado por entrada/salida podemos superponer esperas mediante hilos.

```text
Secuencial:  [req1]──►[req2]──►[req3]
Concurrente: [req1]
             [req2]   varias esperas se superponen
             [req3]
```

Con hilos hablamos de **concurrencia**. En solicitudes HTTP el trabajo puede solaparse porque la mayor parte del tiempo se espera por la red. Esto no implica que todo código Python se ejecute en paralelo sobre varios núcleos.

<a id="threadpool"></a>
### `ThreadPoolExecutor`

`ThreadPoolExecutor` administra un grupo de hilos reutilizables.

| Método | Descripción |
|---|---|
| `ThreadPoolExecutor(max_workers=N)` | Crea un pool de hasta `N` hilos. |
| `submit(fn, *args, **kwargs)` | Programa una llamada y retorna un `Future` inmediatamente. |
| `map(fn, iterable)` | Retorna un iterador; las llamadas se ejecutan concurrentemente y los resultados se entregan en orden de entrada. Obtener el siguiente resultado puede bloquear. |
| `future.result()` | Espera el resultado o vuelve a levantar la excepción. |
| `as_completed(futures)` | Entrega los `Future` a medida que terminan. |

Al salir de `with` se ejecuta `shutdown(wait=True)`, por lo que se espera a los trabajos pendientes.

In [1]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests


BASE = "https://dummyjson.com/"


def obtener_receta(receta_id: int) -> dict:
    respuesta = requests.get(
        f"{BASE}recipes/{receta_id}",
        timeout=10,
    )
    respuesta.raise_for_status()
    return {
        "id": receta_id,
        "status": respuesta.status_code,
        "nombre": respuesta.json().get("name"),
    }

In [2]:
ids_recetas = list(range(1, 11))  # recetas 1 a 10

with ThreadPoolExecutor(max_workers=10) as executor:
    futuros = {executor.submit(obtener_receta, i): i for i in ids_recetas}
    resultados_con = [futuro.result() for futuro in as_completed(futuros)]

for r in sorted(resultados_con, key=lambda x: x["id"]):
    print(f"  Receta {r['id']:2d} [{r['status']}]: {r['nombre']}")


  Receta  1 [200]: Classic Margherita Pizza
  Receta  2 [200]: Vegetarian Stir-Fry
  Receta  3 [200]: Chocolate Chip Cookies
  Receta  4 [200]: Chicken Alfredo Pasta
  Receta  5 [200]: Mango Salsa Chicken
  Receta  6 [200]: Quinoa Salad with Avocado
  Receta  7 [200]: Tomato Basil Bruschetta
  Receta  8 [200]: Beef and Broccoli Stir-Fry
  Receta  9 [200]: Caprese Salad
  Receta 10 [200]: Shrimp Scampi Pasta


<a id="tiempos"></a>
### Comparación de tiempos

La medición utiliza `time.perf_counter()`, apropiado para intervalos. El resultado depende de la red, DNS, carga del servicio, caché y número de hilos; es una observación de esta ejecución, no una aceleración garantizada.

In [3]:
# Forma secuencial
inicio = time.perf_counter()
resultados_sec = [obtener_receta(i) for i in ids_recetas]
tiempo_sec = time.perf_counter() - inicio
print(f"Secuencial  → {tiempo_sec:.2f}s")

# Forma concurrente
inicio = time.perf_counter()
with ThreadPoolExecutor(max_workers=10) as executor:
    futuros = [executor.submit(obtener_receta, i) for i in ids_recetas]
    resultados_con = [f.result() for f in as_completed(futuros)]
tiempo_con = time.perf_counter() - inicio
print(f"Concurrente → {tiempo_con:.2f}s")

if tiempo_con > 0:
    print(f"Razón observada: {tiempo_sec / tiempo_con:.1f}x")

Secuencial  → 7.04s


Concurrente → 0.67s
Razón observada: 10.5x


<a id="race-condition"></a>
## ¿Qué es una condición de carrera?

Una **condición de carrera** ocurre cuando el resultado depende de un orden de ejecución no coordinado entre operaciones concurrentes.

```text
Hilo A: lee 5
Hilo B: lee 5
Hilo A: escribe 6
Hilo B: escribe 6  ← se perdió una actualización
```

Este caso se denomina *lost update*. Las condiciones de carrera pueden afectar memoria compartida, archivos, bases de datos y recursos modificados mediante APIs.

<a id="riesgos-cliente"></a>
## Riesgos de concurrencia en el cliente

<a id="duplicadas"></a>
### Solicitudes duplicadas

Dos clics rápidos en “Comprar” pueden producir dos `POST`. Deshabilitar temporalmente el botón reduce el riesgo, pero la garantía debe implementarse en el servidor.

Un header como `Idempotency-Key` solo funciona si el servidor lo reconoce, guarda el resultado de la primera operación y reutiliza ese resultado ante repeticiones.

```python
import uuid

clave = str(uuid.uuid4())
requests.post(
    "https://api.tienda.com/compras",
    json={"producto_id": 42, "cantidad": 1},
    headers={"Idempotency-Key": clave},
    timeout=10,
)
```

<a id="rate-limit"></a>
### *Rate limiting*

`429 Too Many Requests` indica que el cliente excedió un límite. No es una condición de carrera, pero puede aparecer al aumentar la concurrencia.

Si la respuesta incluye `Retry-After`, el cliente debe respetarlo. En caso contrario puede usar *backoff* exponencial con un pequeño componente aleatorio (*jitter*) para que muchos clientes no reintenten al mismo tiempo.

La función siguiente realiza un intento inicial y hasta `max_reintentos` reintentos. Solo reintenta `429`; para otros errores HTTP utiliza `raise_for_status()`.

In [4]:
import random


def obtener_producto_con_backoff(
    producto_id: int,
    max_reintentos: int = 3,
) -> dict:
    for intento in range(max_reintentos + 1):
        respuesta = requests.get(
            f"{BASE}products/{producto_id}",
            timeout=10,
        )

        if respuesta.status_code != 429:
            respuesta.raise_for_status()
            return {
                "id": producto_id,
                "status": respuesta.status_code,
                "titulo": respuesta.json().get("title"),
            }

        if intento == max_reintentos:
            break

        retry_after = respuesta.headers.get("Retry-After")
        if retry_after is not None and retry_after.isdigit():
            espera = float(retry_after)
        else:
            espera = 2 ** intento + random.uniform(0, 0.5)

        print(
            f"[producto {producto_id}] 429; "
            f"reintento en {espera:.2f}s"
        )
        time.sleep(espera)

    return {
        "id": producto_id,
        "status": 429,
        "titulo": "Sin respuesta tras los reintentos",
    }

#### Simulación con `unittest.mock`

DummyJSON no devuelve `429` fácilmente. La prueba intercepta `requests.get`. Como el contador del mock también es estado compartido, se protege con su propio `Lock`.

In [5]:
import threading
from unittest.mock import MagicMock, patch


_llamadas = {}
_lock_llamadas = threading.Lock()


def mock_get(url, **kwargs):
    producto_id = int(url.split("/")[-1])

    with _lock_llamadas:
        numero_llamada = _llamadas.get(producto_id, 0) + 1
        _llamadas[producto_id] = numero_llamada

    respuesta = MagicMock()
    respuesta.headers = {}

    if producto_id in (2, 4) and numero_llamada == 1:
        respuesta.status_code = 429
        return respuesta
    if producto_id == 2 and numero_llamada == 2:
        respuesta.status_code = 429
        return respuesta

    respuesta.status_code = 200
    respuesta.json.return_value = {
        "title": f"Producto simulado #{producto_id}",
    }
    return respuesta


with patch("requests.get", side_effect=mock_get):
    with ThreadPoolExecutor(max_workers=5) as executor:
        futuros = [
            executor.submit(obtener_producto_con_backoff, i)
            for i in range(1, 6)
        ]
        resultados = [f.result() for f in as_completed(futuros)]

for resultado in sorted(resultados, key=lambda item: item["id"]):
    print(resultado)

[producto 2] 429; reintento en 1.18s
[producto 4] 429; reintento en 1.45s


[producto 2] 429; reintento en 2.21s


{'id': 1, 'status': 200, 'titulo': 'Producto simulado #1'}
{'id': 2, 'status': 200, 'titulo': 'Producto simulado #2'}
{'id': 3, 'status': 200, 'titulo': 'Producto simulado #3'}
{'id': 4, 'status': 200, 'titulo': 'Producto simulado #4'}
{'id': 5, 'status': 200, 'titulo': 'Producto simulado #5'}


<a id="race-servidor"></a>
## Condiciones de carrera en servidores

La concurrencia depende del servidor WSGI utilizado. En el experimento siguiente, Werkzeug atiende cada solicitud en un hilo mediante `threaded=True`.

<a id="estado-compartido"></a>
### Estado compartido sin protección

Cien solicitudes incrementan un contador global. La pausa entre lectura y escritura hace visible la pérdida de actualizaciones.

In [6]:
import logging
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from flask import Flask
from werkzeug.serving import make_server


logging.getLogger("werkzeug").setLevel(logging.ERROR)

app_sin_lock = Flask("contador_sin_lock")
contador_sin_lock = 0


@app_sin_lock.route("/incrementar", methods=["POST"])
def incrementar_sin_lock():
    global contador_sin_lock
    valor = contador_sin_lock
    time.sleep(0.001)
    contador_sin_lock = valor + 1
    return {"ok": True}


@app_sin_lock.route("/contador", methods=["GET"])
def get_contador_sin_lock():
    return {"contador": contador_sin_lock}


srv_sin_lock = make_server(
    "localhost",
    4445,
    app_sin_lock,
    threaded=True,
)
hilo_sin_lock = threading.Thread(
    target=srv_sin_lock.serve_forever,
    daemon=True,
)
hilo_sin_lock.start()

N = 100
with ThreadPoolExecutor(max_workers=20) as executor:
    futuros = [
        executor.submit(
            requests.post,
            "http://localhost:4445/incrementar",
            timeout=5,
        )
        for _ in range(N)
    ]
    for futuro in as_completed(futuros):
        futuro.result().raise_for_status()

resultado = requests.get(
    "http://localhost:4445/contador",
    timeout=5,
).json()["contador"]

srv_sin_lock.shutdown()
srv_sin_lock.server_close()
hilo_sin_lock.join(timeout=2)

print(f"Resultado esperado : {N}")
print(f"Resultado obtenido : {resultado}")
print(f"Valores perdidos   : {N - resultado}")

Resultado esperado : 100
Resultado obtenido : 30
Valores perdidos   : 70


<a id="lock"></a>
### Proteger una sección crítica con `Lock`

`threading.Lock` permite que un solo hilo por vez ejecute la sección crítica. Resuelve este ejemplo dentro de **un proceso**.

En producción pueden existir varios procesos o servidores. Un `Lock` de Python no se comparte entre ellos; para inventarios, reservas o saldos se requieren transacciones, restricciones y operaciones atómicas en la base de datos, o coordinación distribuida.

In [7]:
app_con_lock = Flask("contador_con_lock")
_lock = threading.Lock()
contador_con_lock = 0


@app_con_lock.route("/incrementar", methods=["POST"])
def incrementar_con_lock():
    global contador_con_lock
    with _lock:
        valor = contador_con_lock
        time.sleep(0.001)
        contador_con_lock = valor + 1
    return {"ok": True}


@app_con_lock.route("/contador", methods=["GET"])
def get_contador_con_lock():
    return {"contador": contador_con_lock}


srv_con_lock = make_server(
    "localhost",
    4446,
    app_con_lock,
    threaded=True,
)
hilo_con_lock = threading.Thread(
    target=srv_con_lock.serve_forever,
    daemon=True,
)
hilo_con_lock.start()

with ThreadPoolExecutor(max_workers=20) as executor:
    futuros = [
        executor.submit(
            requests.post,
            "http://localhost:4446/incrementar",
            timeout=5,
        )
        for _ in range(N)
    ]
    for futuro in as_completed(futuros):
        futuro.result().raise_for_status()

resultado = requests.get(
    "http://localhost:4446/contador",
    timeout=5,
).json()["contador"]

srv_con_lock.shutdown()
srv_con_lock.server_close()
hilo_con_lock.join(timeout=2)

print(f"Resultado esperado : {N}")
print(f"Resultado obtenido : {resultado}")
print(f"Valores perdidos   : {N - resultado}")

Resultado esperado : 100
Resultado obtenido : 100
Valores perdidos   : 0
